# Recomendador Peliculas - Series

Paquetes

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from sklearn.neighbors import NearestNeighbors
import pickle
from tqdm import tqdm
import os
import json

Datos

In [2]:
ratings = pd.read_csv(r"..\data_movie_matchmaker\ml_32m\ratings.csv", sep = ",")
ratings.columns = ['user_id', 'movie_id', 'rating', 'date']
ratings['date'] = pd.to_datetime(ratings['date'], unit = 's').dt.date

movies = pd.read_csv(r"..\data_movie_matchmaker\ml_32m\movies.csv", sep = ",")
movies.columns = ["movie_id", "title_year", "genres"]
movies['title'] = movies['title_year'].str.extract(r'^(.*)\s\(\d{4}\)$')[0]
movies['year'] = movies['title_year'].str.extract(r'\((\d{4})\)$')[0]
movies['year'] = movies['year'].replace(np.nan, '1900')
movies.drop(columns = ['title_year'], inplace = True)

print(f"Ratings: {ratings.shape} \n {ratings.head()} \n")
print(f"Movies: {movies.shape} \n {movies.head()} \n")

Ratings: (32000204, 4) 
    user_id  movie_id  rating        date
0        1        17     4.0  1999-12-03
1        1        25     1.0  1999-12-03
2        1        29     2.0  1999-11-22
3        1        30     5.0  1999-12-03
4        1        32     5.0  1999-11-22 

Movies: (87585, 4) 
    movie_id                                       genres  \
0         1  Adventure|Animation|Children|Comedy|Fantasy   
1         2                   Adventure|Children|Fantasy   
2         3                               Comedy|Romance   
3         4                         Comedy|Drama|Romance   
4         5                                       Comedy   

                         title  year  
0                    Toy Story  1995  
1                      Jumanji  1995  
2             Grumpier Old Men  1995  
3            Waiting to Exhale  1995  
4  Father of the Bride Part II  1995   



Funcion de similitud

In [3]:
def build_movie_topk(ratings, k = 100):

    user_codes = ratings['user_id'].astype('category').cat.codes
    movie_codes = ratings['movie_id'].astype('category').cat.codes
    movie_map = dict(enumerate(ratings['movie_id'].astype('category').cat.categories))
    X = csr_matrix((ratings['rating'], (user_codes, movie_codes)))

    model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors = k+1, n_jobs = -1)
    model.fit(X.T)

    distances, indices = model.kneighbors(X.T)
    topk = {}
    for i, (dist, idx) in enumerate(tqdm(zip(distances, indices), total = X.shape[1], desc = "Calculando Top-K")):
        movie_id = movie_map[i]
        neighbors = [(movie_map[j], 1 - d) for d, j in zip(dist[1:], idx[1:])]
        topk[movie_id] = dict(neighbors)

    with open('movie_similarity_topk.pkl', 'wb') as f:
        pickle.dump(topk, f)

    return topk

Matriz de similitud

In [4]:
movie_similarity_topk = build_movie_topk(ratings, k = 100)

Calculando Top-K: 100%|██████████| 84432/84432 [00:04<00:00, 19032.23it/s]


Matriz esparsa

In [5]:
with open("movie_similarity_topk.pkl", "rb") as f:
    movie_similarity_topk = pickle.load(f)

def convert_to_sparse_matrix():
    
    # Obtener todas las movie_ids
    all_movie_ids = sorted(set(movie_similarity_topk.keys()))
    movie_id_to_idx = {mid: idx for idx, mid in enumerate(all_movie_ids)}
    n_movies = len(all_movie_ids)
    
    # Construir matriz sparse
    rows = []
    cols = []
    data = []
    
    for movie_id, neighbors in movie_similarity_topk.items():
        if movie_id not in movie_id_to_idx:
            continue
        row_idx = movie_id_to_idx[movie_id]
        
        for neighbor_id, similarity in neighbors.items():
            if neighbor_id not in movie_id_to_idx:
                continue
            col_idx = movie_id_to_idx[neighbor_id]
            rows.append(row_idx)
            cols.append(col_idx)
            data.append(similarity)
    
    # Crear matriz sparse
    similarity_matrix = csr_matrix(
        (data, (rows, cols)), 
        shape=(n_movies, n_movies),
        dtype=np.float32  # Usar float32 en vez de float64 ahorra 50%
    )
    
    # Guardar matriz sparse (muy comprimida)
    save_npz("movie_similarity.npz", similarity_matrix)
    
    # Guardar mapping de IDs
    with open("movie_id_mapping.json", "w") as f:
        json.dump({
            "id_to_idx": movie_id_to_idx,
            "idx_to_id": {idx: mid for mid, idx in movie_id_to_idx.items()}
        }, f)
    
    print(f"Matriz sparse guardada: movie_similarity.npz")
    print(f"Mapping guardado: movie_id_mapping.json")
    
    # Mostrar tamaños
    original_size = os.path.getsize("movie_similarity_topk.pkl") / (1024*1024)
    new_size = os.path.getsize("movie_similarity.npz") / (1024*1024)
    mapping_size = os.path.getsize("movie_id_mapping.json") / (1024*1024)
    
    print(f"\n Comparación de tamaños:")
    print(f"Original (PKL): {original_size:.2f} MB")
    print(f"Nuevo (NPZ): {new_size:.2f} MB")
    print(f"Mapping (JSON): {mapping_size:.2f} MB")
    print(f"Total nuevo: {new_size + mapping_size:.2f} MB")
    print(f"Ahorro: {original_size - (new_size + mapping_size):.2f} MB ({((original_size - (new_size + mapping_size))/original_size)*100:.1f}%)")

Algoritmo de recomendacion de peliculas

In [6]:
def recommend_movies(movie_similarity_topk, ratings, movies, movies_list, n = 5, min_ratings = 100):

    title_to_id = dict(zip(movies.title, movies.movie_id))
    favorite_ids = [title_to_id[m] for m in movies_list if m in title_to_id]

    scores = {}

    for movie_id in favorite_ids:
        if movie_id not in movie_similarity_topk:
            continue

        for neighbor_id, sim in movie_similarity_topk[movie_id].items():
            scores[neighbor_id] = scores.get(neighbor_id, 0) + sim

    for movie_id in favorite_ids:
        scores.pop(movie_id, None)

    scores = pd.Series(scores).sort_values(ascending=False)

    rating_movies = ratings.groupby('movie_id').agg(rating = ('rating', 'mean'), n_ratings = ('rating', 'count'))
    rating_movies['rating'] = rating_movies['rating'].round(2)
    rating_movies = rating_movies[rating_movies['n_ratings'] >= min_ratings]

    recommended_titles = (movies[movies.movie_id.isin(scores.index)].merge(rating_movies, on = 'movie_id', how = 'inner').head(n)[['title', 'rating', 'n_ratings', 'genres', 'year']].rename(columns={'genres': 'genre'}))

    return recommended_titles

Prueba

In [7]:
movies_list = ["Toy Story", "Cars", "Kung Fu Panda"]
r1 = recommend_movies(movie_similarity_topk, ratings, movies, movies_list)
r1.head()

,title,rating,n_ratings,genre,year
0,Jumanji,3.28,28904,Adventure|Children|Fantasy,1995
1,Twelve Monkeys (a.k.a. 12 Monkeys),3.91,55275,Mystery|Sci-Fi|Thriller,1995
2,Babe,3.59,35969,Children|Drama,1995
3,Seven (a.k.a. Se7en),4.09,63298,Mystery|Thriller,1995
4,"Usual Suspects, The",4.27,67750,Crime|Mystery|Thriller,1995


Lista de peliculas y valoraciones medias

In [8]:
rating_movies = ratings.groupby('movie_id').agg(rating = ('rating', 'mean'), n_ratings = ('rating', 'count'))
rating_movies['rating'] = rating_movies['rating'].round(2)
recommended_titles = movies.merge(rating_movies, on = 'movie_id', how = 'inner')[['movie_id', 'title', 'rating', 'n_ratings', 'genres', 'year']].rename(columns={'genres': 'genre'})
recommended_titles.head()

,movie_id,title,rating,n_ratings,genre,year
0,1,Toy Story,3.90,68997,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,3.28,28904,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,3.14,13134,Comedy|Romance,1995
3,4,Waiting to Exhale,2.85,2806,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,3.06,13154,Comedy,1995


In [9]:
recommended_titles.to_csv('films.csv', index = False)